# Sinhala corpus: production pilot with Surya

Run after merging the required changes into `dev`, or set `BRANCH` to a pushed feature branch containing this notebook. Choose a **T4 GPU** runtime.
This notebook imports staged VM PDFs, processes selected pages and prepares a review packet on Drive. It never approves OCR text or exports training data automatically.

The first run creates a **new deployment store**, preserving the seed catalog but
resetting seed download states before importing the selected VM PDFs and receipts.
Existing stores and raw files are preserved on subsequent runs. Keep one runtime/writer
per store. This is separate from `reviews/ocr/broad_20261011` and its OCR evaluation files.

This run imports LK-EDU-001501 through LK-EDU-001505 and processes pages 1-6
for each document. The VM must first copy its catalog and selected PDF folders with
download receipts to My Drive/sinhala-corpus/vm-import/. Read
`docs/CORPUS_PRODUCTION_GUIDE.md` for limits and approvals.


In [ ]:
# ============================================================
# RUN ORDER:
# 1. Setup variables, mount Drive, clone repo
# 2. Create env, install deps (torch, surya, poppler)
# 3. INIT - Creates store on Drive when it does not exist
# 4. Config update - Copies sources/subjects and repairs Drive paths
# 5. IMPORT - Copies PDFs from vm-import/ to store
# 6. Freeze Surya checkpoints
# 7. Configure processing
# 8. Dry-run download preview
# 9. Actual download (optional)
# 10. PROCESS - OCR selected pages
# 11. Prepare review packet
# 12. Quality report
# ============================================================

from pathlib import Path
import json, subprocess, sys, shutil, datetime
from google.colab import drive

if not shutil.which('nvidia-smi'):
    raise RuntimeError('Choose Runtime → Change runtime type → T4 GPU, then reconnect.')
subprocess.run(['nvidia-smi'], check=True)
drive.mount('/content/drive')
ROOT = Path('/content/sinhala-corpus')
STORE = Path('/content/drive/MyDrive/sinhala-corpus/production_02')
BRANCH = 'dev'
DOC_IDS = ['LK-EDU-001501', 'LK-EDU-001502', 'LK-EDU-001503', 'LK-EDU-001504', 'LK-EDU-001505']
PAGE_NUMBERS = list(range(1, 7))  # pages 1-6 per document = 30 pages total
MAX_PAGES = 30
if ROOT.exists():
    raise RuntimeError('Repository already exists. Use a fresh runtime for setup; resume processing using the later cells.')
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                'https://github.com/Warusha-Shamalka/sinhala-corpus.git', str(ROOT)], check=True)


## Isolated pinned environment

Packages and model weights are downloaded during setup. OCR runs from local weights
in a separate process with network access blocked. Temporary models live in `/content`;
raw documents, processed pages, review packets and audit manifests live on Drive.


In [ ]:
ENV = Path('/content/corpus-env')
if ENV.exists():
    raise RuntimeError('Use a fresh runtime instead of replacing an existing environment.')
subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
subprocess.run([sys.executable,'-m','uv','venv','--python','3.12.12','--seed',str(ENV)],check=True)
PYTHON = str(ENV/'bin/python')
def pip(*args):
    subprocess.run([PYTHON,'-m','pip','install',*args],check=True)
pip('torch==2.7.1','torchvision==0.22.1','--index-url','https://download.pytorch.org/whl/cu126')
pip('PyYAML==6.0.3','surya-ocr==0.17.1','transformers==4.57.6','Pillow==10.4.0')
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','poppler-utils'],check=True)


## Persistent store

This initialization is for a fresh store. It archives the repository seed catalog and
creates a separate acquisition catalog. It does not migrate old raw files or alter the
repository catalog. If `production_02` already exists, its existing catalog is used.
Do not delete the store to resume a run.


In [ ]:
CONFIG = STORE/'configs/pipeline.yaml'
if not STORE.exists():
    subprocess.run([PYTHON,'-B','-m','pipeline.workflow','init','--store',str(STORE)],cwd=ROOT,check=True)
if not CONFIG.is_file():
    raise RuntimeError('Store exists without its configuration. Inspect it; choose a new store only for a new deployment.')


In [ ]:
# Update source/subject configs after init without replacing the Drive-specific pipeline paths.
import shutil
from pathlib import Path

for fname in ['sources.yaml', 'subjects.yaml']:
    shutil.copy(ROOT / 'configs' / fname, STORE / 'configs' / fname)
    print(f"Updated {fname}")

# Repair paths in the Drive config while preserving its processing and download settings.
# This also fixes existing stores if a repo-local pipeline.yaml was copied there earlier.
repair_paths_code = r"""
import sys, yaml
from pathlib import Path
store = Path(sys.argv[1])
path = store / 'configs' / 'pipeline.yaml'
config = yaml.safe_load(path.read_text(encoding='utf-8'))
catalog = str(store / 'corpus/catalog/documents.csv')
raw = str(store / 'corpus/raw')
logs = store / 'corpus/logs'
for section in ('download', 'extraction', 'ocr', 'processing'):
    config[section]['catalog_path'] = catalog
    config[section]['raw_root'] = raw
    config[section]['log_root'] = str(logs / section)
config['extraction']['output_root'] = str(store / 'corpus/extracted/pages')
config['ocr']['extraction_root'] = config['extraction']['output_root']
config['ocr']['output_root'] = str(store / 'corpus/extracted/tesseract')
config['processing']['output_root'] = str(store / 'corpus/extracted/processed')
path.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding='utf-8')
print('Drive download catalog:', config['download']['catalog_path'])
"""
subprocess.run([PYTHON, '-c', repair_paths_code, str(STORE)], cwd=ROOT, check=True)


## Import selected verified PDFs from the VM

Copy the master catalog to `My Drive/sinhala-corpus/vm-import/documents.csv` and the selected document folders to `My Drive/sinhala-corpus/vm-import/pdf/`. This cell validates the catalog row, source approval, receipt, PDF signature, byte count and SHA256 before importing. It preserves existing store rows and refuses conflicting IDs or files.


In [ ]:
from pathlib import Path
import csv, hashlib, json, os, shutil, tempfile
from urllib.parse import urlsplit
import subprocess

IMPORT_ROOT = STORE.parent / 'vm-import'
VM_CATALOG = IMPORT_ROOT / 'documents.csv'
VM_RAW = IMPORT_ROOT / 'pdf'
if not VM_CATALOG.is_file():
    raise FileNotFoundError(f'Copy the VM catalog to {VM_CATALOG} first')
if not VM_RAW.is_dir():
    raise FileNotFoundError(f'Copy the selected VM raw PDF folders under {VM_RAW} first')

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

with VM_CATALOG.open(newline='', encoding='utf-8') as stream:
    reader = csv.DictReader(stream)
    master_fields = reader.fieldnames
    master_rows = list(reader)
if not master_fields or 'doc_id' not in master_fields:
    raise ValueError('VM catalog is missing its doc_id header')
selected_rows = [row for row in master_rows if row.get('doc_id') in DOC_IDS]
if len(selected_rows) != len(DOC_IDS) or len({row['doc_id'] for row in selected_rows}) != len(DOC_IDS):
    raise ValueError(f'Expected exactly one catalog row for each selected ID: {DOC_IDS}')

sources_json = subprocess.check_output([PYTHON, '-c',
    "import json,sys,yaml; print(json.dumps(yaml.safe_load(open(sys.argv[1], encoding='utf-8'))['sources']))",
    str(STORE / 'configs/sources.yaml')], text=True)
sources = json.loads(sources_json)
source_by_name = {item['name']: item for item in sources}
catalog_path = STORE / 'corpus/catalog/documents.csv'
with catalog_path.open(newline='', encoding='utf-8') as stream:
    target_reader = csv.DictReader(stream)
    fields = target_reader.fieldnames
    target_rows = list(target_reader)
if fields != master_fields:
    missing = set(master_fields) - set(fields)
    extra = set(fields) - set(master_fields)
    if missing or extra:
        raise ValueError(f'VM and store catalog schemas differ; missing={sorted(missing)}, extra={sorted(extra)}')
by_id = {row['doc_id']: row for row in target_rows}
raw_root = STORE / 'corpus/raw'
imported = []
for original in selected_rows:
    doc_id = original['doc_id']
    source = source_by_name.get(original.get('source', ''))
    source_url = original.get('source_url', '')
    if not source or not source.get('enabled') or not source.get('verified'):
        raise ValueError(f'{doc_id}: source is not enabled and verified in this repository')
    url = urlsplit(source_url)
    allowed = urlsplit(source.get('base_url', ''))
    if url.scheme != 'https' or not url.hostname or url.hostname != allowed.hostname:
        raise ValueError(f'{doc_id}: source URL does not match the approved HTTPS source host')
    # Redirected download hosts must be explicitly listed in resource_hosts.
    approved_hosts = {allowed.hostname, *source.get('resource_hosts', [])}
    source_dir = VM_RAW / doc_id
    pdfs = sorted(source_dir.glob('*.pdf'))
    if len(pdfs) != 1:
        raise ValueError(f'{doc_id}: expected exactly one staged PDF in {source_dir}; found {len(pdfs)}')
    source_pdf = pdfs[0]
    digest = source_pdf.stem
    source_receipt = source_dir / f'{digest}.json'
    if len(digest) != 64 or not source_receipt.is_file():
        raise ValueError(f'{doc_id}: expected matching SHA256-named JSON receipt beside the PDF')
    receipt = json.loads(source_receipt.read_text(encoding='utf-8'))
    actual_hash = sha256_file(source_pdf)
    size = source_pdf.stat().st_size
    if (actual_hash != digest or receipt.get('schema_version') != 'raw-download-1'
            or receipt.get('doc_id') != doc_id or receipt.get('format') != 'pdf'
            or receipt.get('sha256') != digest or receipt.get('bytes') != size
            or receipt.get('requested_url') != source_url
            or receipt.get('source_id') != source['id']
            or urlsplit(receipt.get('final_url', '')).hostname not in approved_hosts
            or receipt.get('artifact_path') != f'pdf/{doc_id}/{digest}.pdf'):
        raise ValueError(f'{doc_id}: PDF and download receipt do not match the VM catalog')
    with source_pdf.open('rb') as stream:
        start = stream.read(5)
        stream.seek(max(0, size - 2048))
        tail = stream.read()
    if size < 1 or size > 50 * 1024 * 1024 or start != b'%PDF-' or b'%%EOF' not in tail:
        raise ValueError(f'{doc_id}: invalid or over-50-MiB PDF')
    dest_dir = raw_root / 'pdf' / doc_id
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_pdf = dest_dir / f'{digest}.pdf'
    dest_receipt = dest_dir / f'{digest}.json'
    if dest_pdf.exists() and sha256_file(dest_pdf) != digest:
        raise ValueError(f'{doc_id}: destination PDF exists with different contents; refusing overwrite')
    if not dest_pdf.exists():
        shutil.copyfile(source_pdf, dest_pdf)
    receipt.update(artifact_path=f'pdf/{doc_id}/{digest}.pdf', local_filename=str(dest_pdf))
    encoded_receipt = json.dumps(receipt, ensure_ascii=False, indent=2) + '\n'
    if dest_receipt.exists() and dest_receipt.read_text(encoding='utf-8') != encoded_receipt:
        old = json.loads(dest_receipt.read_text(encoding='utf-8'))
        if old.get('sha256') != digest or old.get('doc_id') != doc_id:
            raise ValueError(f'{doc_id}: destination receipt conflicts; refusing overwrite')
    else:
        dest_receipt.write_text(encoded_receipt, encoding='utf-8')
    row = dict(original)
    row.update(status='DOWNLOADED', local_filename=str(dest_pdf), sha256=digest,
               extraction_method='', ocr_required='', page_count='', quality_score='',
               pipeline_version=receipt.get('pipeline_version', 'download-0.1'), error='')
    existing = by_id.get(doc_id)
    if existing:
        for key in ('title', 'source', 'source_url'):
            if existing.get(key) != row.get(key):
                raise ValueError(f'{doc_id}: existing Drive catalog has conflicting {key}; inspect before continuing')
        if existing.get('sha256') and existing['sha256'] != digest:
            raise ValueError(f'{doc_id}: existing Drive catalog has a different PDF version; refusing replacement')
        if not existing.get('sha256'):
            existing.update(row)
    else:
        target_rows.append(row)
        by_id[doc_id] = row
    imported.append({'doc_id': doc_id, 'sha256': digest, 'bytes': size, 'path': str(dest_pdf)})

# Single Colab writer: replace the CSV only after every selected PDF has validated.
fd, temp_name = tempfile.mkstemp(prefix='.documents-', suffix='.csv', dir=catalog_path.parent)
try:
    with os.fdopen(fd, 'w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(target_rows)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temp_name, catalog_path)
finally:
    Path(temp_name).unlink(missing_ok=True)
print(json.dumps({'imported_or_verified': imported, 'catalog': str(catalog_path)}, ensure_ascii=False, indent=2))


## Freeze Surya checkpoints and run environment

Surya does not require a Hugging Face token. Its pinned detector and recognizer are
verified by SHA256. Changing an OCR backend/revision creates separate page artifacts;
see the guide before installing a different backend's dependencies.


In [ ]:
MODEL_PATH = Path('/content/surya-models')
selection_code = r"""
import json,sys
from pathlib import Path
from pipeline.ocr.surya_backend import prepare_models,verify_models
from pipeline.download.downloader import file_hash
root=Path(sys.argv[1]); output=Path(sys.argv[2])
if root.exists():
    revision=file_hash(root/'selection.json')
    result={**verify_models(root,revision),'path':str(root),'revision':revision,
            'revision_kind':'local_components_manifest_sha256'}
else:
    result=prepare_models(root)
output.write_text(json.dumps(result,indent=2)+'\n')
"""
SELECTION = Path('/content/corpus_model_selection.json')
subprocess.run([PYTHON,'-c',selection_code,str(MODEL_PATH),str(SELECTION)],cwd=ROOT,check=True)
selection = json.loads(SELECTION.read_text())
manifest_root = STORE/'model_manifests'
manifest_root.mkdir(exist_ok=True)
model_manifest = manifest_root/(selection['revision']+'.json')
if model_manifest.exists():
    if json.loads(model_manifest.read_text()) != selection:
        raise RuntimeError('Saved model manifest differs; inspect it before continuing.')
else:
    model_manifest.write_text(json.dumps(selection,indent=2)+'\n')
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
(manifest_root/f'requirements_{stamp}.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
(manifest_root/f'repository_commit_{stamp}.txt').write_text(subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True))


In [ ]:
configure_code = r"""
import sys,yaml
from pathlib import Path
path=Path(sys.argv[1]);config=yaml.safe_load(path.read_text())
config['processing']['ocr'].update(backend='surya',model_path=sys.argv[2],revision=sys.argv[3])
config['processing'].update(max_pages=int(sys.argv[4]),max_documents=1,ocr_mode='all')
config['download']['max_documents']=1
path.write_text(yaml.safe_dump(config,sort_keys=False,allow_unicode=True))
"""
subprocess.run([PYTHON,'-c',configure_code,str(CONFIG),str(MODEL_PATH),selection['revision'],str(MAX_PAGES)],cwd=ROOT,check=True)

def execute(*args):
    # Show diagnostics from both streams if a workflow command fails.
    result = subprocess.run([PYTHON, '-B', '-m', 'pipeline.workflow', *args],
                            cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode:
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()
    if result.stderr:
        print(result.stderr)
    print(result.stdout)
    return json.loads(result.stdout)

def selection_args():
    return [part for doc in DOC_IDS for part in ['--doc-id',doc]]


## Download preview, then the bounded download

The import cell registered the staged VM PDF and receipt. This bounded collect step
verifies the imported receipt and checkpoints the record without fetching a second copy.
For future new resources, use the persistent-catalog discovery command in the guide.


In [ ]:
execute('collect','--config',str(CONFIG),*selection_args(),'--dry-run')


In [ ]:
download_report = execute('collect','--config',str(CONFIG),*selection_args(),'--progress','plain')


## Process selected pages

Each page is saved to Drive immediately. `predicted` means generation completed;
it does not mean the text is accurate. Failed/incomplete pages and warnings remain visible.
Rerun this cell to verify existing pages and process remaining selected pages within the
batch cap. To expand, change `PAGE_NUMBERS`/`MAX_PAGES`, rerun the configuration cell and
this cell. Keep the same store. For retrying a failed OCR output, use a new output root/run
variant as described in the guide; existing artifacts are immutable.


In [ ]:
page_args = [] if PAGE_NUMBERS is None else [part for page in PAGE_NUMBERS for part in ['--page-number',str(page)]]
processing_report = execute('process','--config',str(CONFIG),*selection_args(),*page_args,'--progress','plain')
execute('status','--config',str(CONFIG))


## Prepare the manual review packet

A new review folder contains `index.html`, source page images, editable `texts/*.txt`,
`review.csv` and provenance. Download the complete packet from Drive and open `index.html`
locally, or view individual images and text files in Drive. The index shows the initial
prediction; edited text files are the authoritative correction drafts.

Have one teammate correct the text and a different teammate verify the full source,
equations and reading order. Do not approve unreadable material. Approval/export commands
are in the guide and are deliberately not executed by this notebook.


In [ ]:
review_stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
REVIEW = STORE/'corpus/clean/reviews'/f'surya_{review_stamp}'
execute('prepare-review','--config',str(CONFIG),'--output',str(REVIEW),
        '--config-hash',processing_report['processing_config_sha256'])
print('Review packet:', REVIEW)


## Review coverage and quality report

This offline report identifies pending review, OCR failures, invalid approvals and split/duplicate blockers.
All newly prepared pages are pending: that is expected. Nothing is approved by this cell.
After editing and manually approving source-checked corrections, rerun this cell to save a new report snapshot.
Correction distances measure editing burden, not independent OCR accuracy. Read `docs/CORPUS_QUALITY_GUIDE.md`.


In [ ]:
quality_stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
QUALITY = STORE/'corpus/logs/quality'/f'surya_{quality_stamp}'
quality_report = execute('quality','--packet',str(REVIEW),'--output',str(QUALITY))
print('Read the summary on Drive:', QUALITY/'report.md')


## Optional: continue acquisition while review is deferred

You can collect provisional OCR text now without retyping every page. No accuracy percentage is certified.
Set `RUN_MORE = True` below to process the selected document in bounded cycles. This selects the whole PDF,
including pages beyond the earlier two-page trial; completed artifacts are verified and reused.
The saved text remains unreviewed, with warnings and `training_approved=false`. It is not a reviewed training release.
Start with ten new pages and read `docs/BATCH_COLLECTION_GUIDE.md` before increasing the caps.


In [ ]:
RUN_MORE = False
batch_report = None
if RUN_MORE:
    batch_report = execute('run','--config',str(CONFIG),*selection_args(),
                           '--max-batches','5','--max-download-jobs','1',
                           '--max-new-pages','10','--progress','plain')
else:
    print('Set RUN_MORE=True to run the bounded provisional collection batch.')


In [ ]:
if batch_report and batch_report.get('processed_bundles'):
    provisional_stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    PROVISIONAL = STORE/'corpus/clean/provisional'/f'surya_{provisional_stamp}'
    execute('provisional','--run-report',batch_report['report_path'],'--output',str(PROVISIONAL))
    print('Unreviewed text snapshot:', PROVISIONAL/'provisional_text.jsonl')


## Optional: daily training release

Build filtered experimental text candidates from the provisional snapshot. No model training or submission runs here.
Set `BUILD_DAILY=True`. For the first release leave `LAST_CONSUMED_RELEASE=None`. Later, use the release actually
consumed by the checkpoint you are continuing from, not automatically the latest folder.
Use `train.jsonl` for a fresh base and `added_train.jsonl` for a matching continual checkpoint.
Incomplete/warning-bearing OCR is quarantined. Read `docs/DAILY_TRAINING_RELEASE_GUIDE.md`.


In [ ]:
BUILD_DAILY = False
LAST_CONSUMED_RELEASE = None
if BUILD_DAILY:
    if 'PROVISIONAL' not in globals() or not (PROVISIONAL/'manifest.json').is_file():
        raise RuntimeError('Create the provisional snapshot first, or set PROVISIONAL to its actual folder.')
    daily_stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    DAILY = STORE/'datasets/pretraining'/f'daily_{daily_stamp}'
    daily_args = ['--provisional',str(PROVISIONAL),'--output',str(DAILY)]
    if LAST_CONSUMED_RELEASE is not None: daily_args += ['--previous',str(LAST_CONSUMED_RELEASE)]
    daily_report = execute('daily-release',*daily_args)
    execute('verify-daily-release','--root',str(DAILY))
    print('Training handoff:', DAILY/'handoff.json')
    print('Release counts:', daily_report['counts'])
else:
    print('Set BUILD_DAILY=True when you want to freeze training candidates.')
